In [1]:
import torch
import torch.nn as nn

torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

DATA PREPARATION!

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

# dataset
df = pd.read_csv("fmnist_small.csv")

# split train and test
X = df.iloc[:,1:]
y = df.iloc[:,0]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=24)

# Scale only the pixels not the labels!
X_train_scaled = X_train/255.0
X_test_scaled = X_test/255.0

# Convert all to tensors
X_train_tensor = torch.tensor(X_train_scaled.values, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled.values, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.long)
y_test_tensor  = torch.tensor(y_test.values, dtype=torch.long)

DATA LOADING!

In [3]:
from torch.utils.data import Dataset, DataLoader

class CustomDataset(Dataset):
    def __init__(self, X_train_tensor, y_train_tensor):
        self.X_train_tensor = X_train_tensor
        self.y_train_tensor = y_train_tensor
    def __len__(self):
        return len(self.X_train_tensor)
    def __getitem__(self,idx):
        return self.X_train_tensor[idx], self.y_train_tensor[idx]

train_dataset = CustomDataset(X_train_tensor, y_train_tensor)
test_dataset = CustomDataset(X_test_tensor, y_test_tensor)

MODEL ARCHITECTURE!

In [4]:
class MyNN(nn.Module):
  def __init__(self, input_dim, output_dim, num_hidden_layers, neurons_per_layer, dropout_rate):
    super().__init__()
    layers = []
    for i in range(num_hidden_layers):
        layers.append(nn.Linear(input_dim, neurons_per_layer)),
        layers.append(nn.BatchNorm1d(neurons_per_layer)),
        layers.append(nn.ReLU()),
        layers.append(nn.Dropout(dropout_rate)),
        input_dim = neurons_per_layer
    layers.append(nn.Linear(neurons_per_layer, output_dim))
    self.model = nn.Sequential(*layers)

  def forward(self, x):
    return self.model(x)

OBJECTIVE FUNCTION!

In [5]:
import torch.optim as optim

def objective(trial):
    # next hyperparameter values from the search space
    num_hidden_layers = trial.suggest_int("num_hidden_layers", 1, 5)
    neurons_per_layer = trial.suggest_int("neurons_per_layer", 8, 128, step=8)
    epochs = trial.suggest_int("epochs", 10, 50, step=10)
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-1, log=True)
    dropout_rate = trial.suggest_float("dropout_rate", 0.1, 0.5, step=0.1)
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64, 128])
    optimizer_name = trial.suggest_categorical("optimizer", ['Adam', 'SGD', 'RMSprop'])
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-3, log=True)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True, pin_memory=True)

    # model init
    input_dim = 784
    output_dim = 10

    # initialize the model
    model = MyNN(input_dim, output_dim, num_hidden_layers, neurons_per_layer, dropout_rate)
    model.to(device)

    # loss and optimizer selection
    loss_fun = nn.CrossEntropyLoss()
    if optimizer_name == 'Adam':
        optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    elif optimizer_name == 'SGD':
        optimizer = optim.SGD(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    elif optimizer_name == 'RMSprop':
        optimizer = optim.RMSprop(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

    # training loop
    for epoch in range(epochs):
        for batch_features, batch_labels in train_loader:
            optimizer.zero_grad()
            # move data to gpu
            batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
            # forward pass
            outputs = model(batch_features)
            # calculate loss
            loss = loss_fun(outputs, batch_labels)
            # back pass
            loss.backward()
            # update grads
            optimizer.step()

    # evaluation
    model.eval()
    # evaluation on test data
    total = len(y_test_tensor)
    correct = 0
    with torch.no_grad():
        for batch_features, batch_labels in test_loader:
            batch_features, batch_labels = batch_features.to(device), batch_labels.to(device)
            outputs = model(batch_features)
            logit_values, predicted = torch.max(outputs, 1)
            correct = correct + (predicted == batch_labels).sum().item()
        accuracy = correct/total

    return accuracy

In [6]:
# !pip install optuna

In [7]:
import optuna

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=10)

[I 2026-08-23 01:37:10,843] A new study created in memory with name: no-name-1ed2f571-4a97-4b89-970b-12cc7792e0bf
[I 2026-08-23 01:37:38,940] Trial 0 finished with value: 0.8225 and parameters: {'num_hidden_layers': 3, 'neurons_per_layer': 128, 'epochs': 30, 'learning_rate': 0.012237241288640694, 'dropout_rate': 0.1, 'batch_size': 32, 'optimizer': 'Adam', 'weight_decay': 6.903682454521273e-05}. Best is trial 0 with value: 0.8225.
[I 2026-08-23 01:38:30,309] Trial 1 finished with value: 0.8358333333333333 and parameters: {'num_hidden_layers': 2, 'neurons_per_layer': 40, 'epochs': 40, 'learning_rate': 0.00327894501497424, 'dropout_rate': 0.1, 'batch_size': 16, 'optimizer': 'RMSprop', 'weight_decay': 3.702215598211285e-05}. Best is trial 1 with value: 0.8358333333333333.
[I 2026-08-23 01:38:56,747] Trial 2 finished with value: 0.8491666666666666 and parameters: {'num_hidden_layers': 2, 'neurons_per_layer': 128, 'epochs': 20, 'learning_rate': 0.00011142116036937712, 'dropout_rate': 0.2, 'b

In [18]:
print(study.best_value)
print(study.best_params)
print(study.best_trial)

print(study.best_trial.number)
print(study.best_trial.value)
print(study.best_trial.params)

study.trials

0.8491666666666666
{'num_hidden_layers': 2, 'neurons_per_layer': 128, 'epochs': 20, 'learning_rate': 0.00011142116036937712, 'dropout_rate': 0.2, 'batch_size': 16, 'optimizer': 'Adam', 'weight_decay': 5.323334374798247e-05}
FrozenTrial(number=2, state=<TrialState.COMPLETE: 1>, values=[0.8491666666666666], datetime_start=datetime.datetime(2026, 8, 23, 1, 38, 30, 311368), datetime_complete=datetime.datetime(2026, 8, 23, 1, 38, 56, 747333), params={'num_hidden_layers': 2, 'neurons_per_layer': 128, 'epochs': 20, 'learning_rate': 0.00011142116036937712, 'dropout_rate': 0.2, 'batch_size': 16, 'optimizer': 'Adam', 'weight_decay': 5.323334374798247e-05}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'num_hidden_layers': IntDistribution(high=5, log=False, low=1, step=1), 'neurons_per_layer': IntDistribution(high=128, log=False, low=8, step=8), 'epochs': IntDistribution(high=50, log=False, low=10, step=10), 'learning_rate': FloatDistribution(high=0.1, log=True, low=1e-05

[FrozenTrial(number=0, state=<TrialState.COMPLETE: 1>, values=[0.8225], datetime_start=datetime.datetime(2026, 8, 23, 1, 37, 10, 846947), datetime_complete=datetime.datetime(2026, 8, 23, 1, 37, 38, 939889), params={'num_hidden_layers': 3, 'neurons_per_layer': 128, 'epochs': 30, 'learning_rate': 0.012237241288640694, 'dropout_rate': 0.1, 'batch_size': 32, 'optimizer': 'Adam', 'weight_decay': 6.903682454521273e-05}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'num_hidden_layers': IntDistribution(high=5, log=False, low=1, step=1), 'neurons_per_layer': IntDistribution(high=128, log=False, low=8, step=8), 'epochs': IntDistribution(high=50, log=False, low=10, step=10), 'learning_rate': FloatDistribution(high=0.1, log=True, low=1e-05, step=None), 'dropout_rate': FloatDistribution(high=0.5, log=False, low=0.1, step=0.1), 'batch_size': CategoricalDistribution(choices=(16, 32, 64, 128)), 'optimizer': CategoricalDistribution(choices=('Adam', 'SGD', 'RMSprop')), 'weight_

In [19]:
df = study.trials_dataframe()
print(df)

   number     value             datetime_start          datetime_complete  \
0       0  0.822500 2026-08-23 01:37:10.846947 2026-08-23 01:37:38.939889   
1       1  0.835833 2026-08-23 01:37:38.942457 2026-08-23 01:38:30.309357   
2       2  0.849167 2026-08-23 01:38:30.311368 2026-08-23 01:38:56.747333   
3       3  0.770833 2026-08-23 01:38:56.748743 2026-08-23 01:39:09.817719   
4       4  0.683333 2026-08-23 01:39:09.819209 2026-08-23 01:39:42.783170   
5       5  0.820000 2026-08-23 01:39:42.787236 2026-08-23 01:40:08.334203   
6       6  0.824167 2026-08-23 01:40:08.336148 2026-08-23 01:40:39.033927   
7       7  0.812500 2026-08-23 01:40:39.037615 2026-08-23 01:40:48.181235   
8       8  0.841667 2026-08-23 01:40:48.185722 2026-08-23 01:41:03.529899   
9       9  0.803333 2026-08-23 01:41:03.532669 2026-08-23 01:41:32.170062   

                duration  params_batch_size  params_dropout_rate  \
0 0 days 00:00:28.092942                 32                  0.1   
1 0 days 00:00:5